# 311 — Program Robustness

## Objective

Evaluate the robustness of the phenotype associations identified for the transcriptomic candidate programs discovered in notebook 310.

The analysis is restricted to the 10 ICA programs previously classified as discovery-level phenotype-associated candidates. No new programs are discovered, no transcriptomic representation is redefined, and the primary pharmacological phenotype remains the model-level median raw `LN_IC50` selected upstream.

This notebook evaluates whether the observed program–phenotype associations remain interpretable under prespecified stress tests addressing:

- lineage dependence of program scores;
- lineage-adjusted program–phenotype associations;
- leave-one-lineage-out sensitivity;
- lineage-stratified resampling;
- alternative phenotype representations frozen in notebook 309;
- drug-response coverage;
- available biological, culture, provenance, and technical cell-line covariates;
- lineage-preserving permutation controls.

## Scope

This notebook evaluates **association robustness**, not representation stability.

ICA and NMF multiseed stability, program recoverability, ICA–NMF structural comparison, and candidate definition were completed in notebook 310 and are not repeated here.

The 10 candidate programs constitute a frozen analysis set. Robustness analyses cannot add candidates, replace ICA programs with NMF factors, redefine the primary phenotype, or select the representation producing the strongest pharmacological association.

Cross-method support is retained as complementary metadata and is not used as a mandatory robustness criterion.

## Methodological interpretation

All robustness analyses are performed on the same frozen 713-model DepMap–GDSC cohort used for discovery.

Because the candidate programs were selected using their association with the primary phenotype in this cohort, the analyses in this notebook are internal post-selection stress tests rather than independent validation.

Accordingly, robustness is evaluated primarily through preservation of effect magnitude and direction, sensitivity to lineage and prespecified covariates, and resistance to influential lineage structure. Statistical significance alone is not sufficient for candidate promotion.

Association does not imply causality. The pharmacological phenotype represents a relative baseline resistance-like context in GDSC and must not be interpreted as clinical or acquired drug resistance.

## Boundary with Phase 4

This notebook does not compare cell-line programs with TCGA tumor programs and does not construct consensus programs.

Tumor–cell-line matching, multiview cross-system recoverability, structural-family accounting, consensus construction, and tumor-specific rules derived from the exploratory audit belong to Phase 4.

The audit informs the robustness safeguards used here without using tumor-program identities to select or prioritize cell-line candidates.

---

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

PHARMACOLOGY_DIR = Paths.pharmacology
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs

PHENOTYPE_SENSITIVITY_PATH = (
    PHARMACOLOGY_DIR
    / "309_phenotype_sensitivity_representations.parquet"
)

PROGRAM_SCORES_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_scores.parquet"
)

PROGRAM_ASSOCIATIONS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_program_phenotype_associations.csv"
)

DISCOVERY_METADATA_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_program_discovery_metadata.json"
)

OUTPUT_DIR = CELL_LINE_PROGRAM_DIR

In [3]:
# =============================================================================
# Load authoritative robustness inputs
# =============================================================================

phenotype_sensitivity = pd.read_parquet(
    PHENOTYPE_SENSITIVITY_PATH
)

ica_program_scores = pd.read_parquet(
    PROGRAM_SCORES_PATH
)

program_associations = pd.read_csv(
    PROGRAM_ASSOCIATIONS_PATH
)

with DISCOVERY_METADATA_PATH.open("r", encoding="utf-8") as handle:
    discovery_metadata = json.load(handle)

print("Phenotype sensitivity :", phenotype_sensitivity.shape)
print("ICA program scores    :", ica_program_scores.shape)
print("Program associations  :", program_associations.shape)

Phenotype sensitivity : (713, 16)
ICA program scores    : (713, 51)
Program associations  : (50, 24)


In [4]:
# =============================================================================
# Freeze discovery-level candidate programs
# =============================================================================

candidate_programs = (
    program_associations
    .loc[program_associations["q_value"].lt(0.05)]
    .copy()
)

candidate_program_ids = candidate_programs[
    "program_id"
].tolist()

print("Frozen candidate programs:", len(candidate_program_ids))
print(candidate_program_ids)

Frozen candidate programs: 10
['ICA_PROGRAM_06', 'ICA_PROGRAM_07', 'ICA_PROGRAM_09', 'ICA_PROGRAM_13', 'ICA_PROGRAM_18', 'ICA_PROGRAM_20', 'ICA_PROGRAM_29', 'ICA_PROGRAM_33', 'ICA_PROGRAM_42', 'ICA_PROGRAM_46']


In [5]:
# =============================================================================
# Construct candidate-program robustness table
# =============================================================================

robustness_data = (
    phenotype_sensitivity
    .merge(
        ica_program_scores[
            ["ModelID", *candidate_program_ids]
        ],
        on="ModelID",
        how="inner",
    )
)

print("Robustness analysis table:", robustness_data.shape)

Robustness analysis table: (713, 26)


In [6]:
# =============================================================================
# Quantify lineage dependence of candidate program scores
# =============================================================================

lineage_eta_squared = []

for program_id in candidate_program_ids:
    values = robustness_data[program_id]
    grand_mean = values.mean()

    ss_total = ((values - grand_mean) ** 2).sum()

    ss_between = (
        robustness_data
        .groupby("OncotreeLineage")[program_id]
        .agg(["size", "mean"])
        .assign(
            weighted_ss=lambda x:
                x["size"] * (x["mean"] - grand_mean) ** 2
        )["weighted_ss"]
        .sum()
    )

    lineage_eta_squared.append(
        {
            "program_id": program_id,
            "lineage_eta_squared": ss_between / ss_total,
        }
    )

lineage_eta_squared = (
    pd.DataFrame(lineage_eta_squared)
    .sort_values("lineage_eta_squared", ascending=False)
    .reset_index(drop=True)
)

lineage_eta_squared

,program_id,lineage_eta_squared
0,ICA_PROGRAM_18,0.555058
1,ICA_PROGRAM_29,0.450314
2,ICA_PROGRAM_09,0.303030
3,ICA_PROGRAM_06,0.242616
4,ICA_PROGRAM_42,0.137175
5,ICA_PROGRAM_20,0.118799
6,ICA_PROGRAM_33,0.107989
7,ICA_PROGRAM_07,0.104019
8,ICA_PROGRAM_13,0.075770
9,ICA_PROGRAM_46,0.020955


In [7]:
# =============================================================================
# Construct lineage-adjusted candidate program scores
# =============================================================================

lineage_program_means = (
    robustness_data
    .groupby("OncotreeLineage")[candidate_program_ids]
    .transform("mean")
)

for program_id in candidate_program_ids:
    robustness_data[
        f"{program_id}_lineage_residual"
    ] = (
        robustness_data[program_id]
        - lineage_program_means[program_id]
    )

In [8]:
# =============================================================================
# Evaluate primary lineage-adjusted program associations
# =============================================================================

lineage_adjusted_associations = []

for program_id in candidate_program_ids:
    adjusted_rho, _ = spearmanr(
        robustness_data[f"{program_id}_lineage_residual"],
        robustness_data["selected_phenotype_lineage_residual"],
    )

    discovery_rho = candidate_programs.loc[
        candidate_programs["program_id"].eq(program_id),
        "spearman_rho",
    ].iloc[0]

    lineage_adjusted_associations.append(
        {
            "program_id": program_id,
            "discovery_rho": discovery_rho,
            "lineage_adjusted_rho": adjusted_rho,
            "absolute_rho_change": abs(adjusted_rho) - abs(discovery_rho),
            "effect_retention": (
                abs(adjusted_rho) / abs(discovery_rho)
            ),
        }
    )

lineage_adjusted_associations = (
    pd.DataFrame(lineage_adjusted_associations)
    .sort_values(
        "lineage_adjusted_rho",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

lineage_adjusted_associations

,program_id,discovery_rho,lineage_adjusted_rho,absolute_rho_change,effect_retention
0,ICA_PROGRAM_46,0.211964,0.245619,0.033655,1.158776
1,ICA_PROGRAM_06,0.129352,0.149012,0.019660,1.151987
2,ICA_PROGRAM_13,-0.112589,-0.136528,0.023939,1.212625
3,ICA_PROGRAM_07,0.100847,0.109690,0.008843,1.087690
4,ICA_PROGRAM_20,0.103747,0.099840,-0.003907,0.962338
5,ICA_PROGRAM_29,-0.132617,-0.097379,-0.035238,0.734284
6,ICA_PROGRAM_33,-0.145618,-0.075251,-0.070367,0.516772
7,ICA_PROGRAM_42,-0.132021,-0.051702,-0.080319,0.391622
8,ICA_PROGRAM_09,-0.192371,-0.038606,-0.153765,0.200686
9,ICA_PROGRAM_18,-0.096073,-0.000377,-0.095696,0.003925


In [9]:
# =============================================================================
# Compute leave-one-lineage-out associations
# =============================================================================

lolo_associations = []

for lineage in robustness_data["OncotreeLineage"].unique():
    lineage_mask = robustness_data["OncotreeLineage"].ne(lineage)

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            robustness_data.loc[
                lineage_mask,
                f"{program_id}_lineage_residual",
            ],
            robustness_data.loc[
                lineage_mask,
                "selected_phenotype_lineage_residual",
            ],
        )

        lolo_associations.append(
            {
                "program_id": program_id,
                "left_out_lineage": lineage,
                "spearman_rho": rho,
            }
        )

lolo_associations = pd.DataFrame(lolo_associations)

In [10]:
# =============================================================================
# Summarize leave-one-lineage-out association stability
# =============================================================================

lolo_summary_rows = []

for program_id in candidate_program_ids:
    program_lolo = lolo_associations.loc[
        lolo_associations["program_id"].eq(program_id)
    ].copy()

    full_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    program_lolo["absolute_deviation"] = (
        program_lolo["spearman_rho"] - full_rho
    ).abs()

    most_influential = program_lolo.loc[
        program_lolo["absolute_deviation"].idxmax()
    ]

    lolo_summary_rows.append(
        {
            "program_id": program_id,
            "lineage_adjusted_rho": full_rho,
            "lolo_min_rho": program_lolo["spearman_rho"].min(),
            "lolo_max_rho": program_lolo["spearman_rho"].max(),
            "max_absolute_deviation": most_influential["absolute_deviation"],
            "most_influential_lineage": most_influential["left_out_lineage"],
            "direction_preserved_fraction": (
                np.sign(program_lolo["spearman_rho"])
                .eq(np.sign(full_rho))
                .mean()
            ),
        }
    )

lolo_summary = (
    pd.DataFrame(lolo_summary_rows)
    .sort_values("max_absolute_deviation", ascending=False)
    .reset_index(drop=True)
)

lolo_summary

,program_id,lineage_adjusted_rho,lolo_min_rho,lolo_max_rho,max_absolute_deviation,most_influential_lineage,direction_preserved_fraction
0,ICA_PROGRAM_33,-0.075251,-0.139368,-0.049951,0.064117,Lymphoid,1.000000
1,ICA_PROGRAM_09,-0.038606,-0.049844,-0.007234,0.031372,Breast,1.000000
2,ICA_PROGRAM_18,-0.000377,-0.021796,0.029934,0.030311,Lung,0.555556
3,ICA_PROGRAM_46,0.245619,0.215501,0.262006,0.030118,Lung,1.000000
4,ICA_PROGRAM_20,0.099840,0.073390,0.118257,0.026450,Bowel,1.000000
5,ICA_PROGRAM_42,-0.051702,-0.069475,-0.027035,0.024667,Lung,1.000000
6,ICA_PROGRAM_29,-0.097379,-0.117041,-0.083736,0.019662,Lymphoid,1.000000
7,ICA_PROGRAM_13,-0.136528,-0.148477,-0.118438,0.018090,Head and Neck,1.000000
8,ICA_PROGRAM_06,0.149012,0.136929,0.156354,0.012083,Esophagus/Stomach,1.000000
9,ICA_PROGRAM_07,0.109690,0.097705,0.120224,0.011986,Breast,1.000000


In [11]:
# =============================================================================
# Resampling parameters
# =============================================================================

N_BOOTSTRAP = 2000
RANDOM_SEED = 20260813

rng = np.random.default_rng(RANDOM_SEED)

In [12]:
# =============================================================================
# Compute lineage-stratified bootstrap associations
# =============================================================================

bootstrap_columns = [
    "selected_phenotype",
    *candidate_program_ids,
]

lineage_indices = [
    group.index.to_numpy()
    for _, group in robustness_data.groupby(
        "OncotreeLineage",
        sort=False,
    )
]

bootstrap_associations = []

for bootstrap_id in range(N_BOOTSTRAP):
    sampled_index = np.concatenate(
        [
            rng.choice(
                indices,
                size=len(indices),
                replace=True,
            )
            for indices in lineage_indices
        ]
    )

    bootstrap_sample = robustness_data.loc[
        sampled_index,
        ["OncotreeLineage", *bootstrap_columns],
    ].copy()

    bootstrap_residuals = (
        bootstrap_sample[bootstrap_columns]
        - bootstrap_sample
        .groupby("OncotreeLineage")[bootstrap_columns]
        .transform("mean")
    )

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            bootstrap_residuals[program_id],
            bootstrap_residuals["selected_phenotype"],
        )

        bootstrap_associations.append(
            {
                "bootstrap_id": bootstrap_id,
                "program_id": program_id,
                "spearman_rho": rho,
            }
        )

bootstrap_associations = pd.DataFrame(
    bootstrap_associations
)

print(
    "Bootstrap iterations completed:",
    bootstrap_associations["bootstrap_id"].nunique(),
)

Bootstrap iterations completed: 2000


In [13]:
# =============================================================================
# Summarize lineage-stratified bootstrap stability
# =============================================================================

bootstrap_summary = (
    bootstrap_associations
    .groupby("program_id")["spearman_rho"]
    .agg(
        bootstrap_median="median",
        ci_lower=lambda x: x.quantile(0.025),
        ci_upper=lambda x: x.quantile(0.975),
    )
    .reset_index()
    .merge(
        candidate_programs[
            ["program_id", "spearman_rho"]
        ].rename(
            columns={"spearman_rho": "discovery_rho"}
        ),
        on="program_id",
        how="left",
    )
)

direction_preservation = (
    bootstrap_associations
    .merge(
        bootstrap_summary[
            ["program_id", "discovery_rho"]
        ],
        on="program_id",
        how="left",
    )
    .assign(
        direction_preserved=lambda x:
            np.sign(x["spearman_rho"])
            == np.sign(x["discovery_rho"])
    )
    .groupby("program_id")["direction_preserved"]
    .mean()
    .rename("direction_preserved_fraction")
    .reset_index()
)

bootstrap_summary = (
    bootstrap_summary
    .merge(
        direction_preservation,
        on="program_id",
        how="left",
    )
    .sort_values(
        "bootstrap_median",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

bootstrap_summary

,program_id,bootstrap_median,ci_lower,ci_upper,discovery_rho,direction_preserved_fraction
0,ICA_PROGRAM_46,0.246309,0.170688,0.312244,0.211964,1.0000
1,ICA_PROGRAM_06,0.146251,0.067667,0.221131,0.129352,0.9995
2,ICA_PROGRAM_13,-0.140025,-0.216313,-0.064684,-0.112589,1.0000
3,ICA_PROGRAM_07,0.110435,0.033075,0.184457,0.100847,0.9980
4,ICA_PROGRAM_20,0.100646,0.025245,0.175991,0.103747,0.9960
5,ICA_PROGRAM_29,-0.094252,-0.168401,-0.018720,-0.132617,0.9910
6,ICA_PROGRAM_33,-0.071539,-0.146266,0.001734,-0.145618,0.9710
7,ICA_PROGRAM_42,-0.057849,-0.130857,0.010930,-0.132021,0.9475
8,ICA_PROGRAM_09,-0.040400,-0.111177,0.030173,-0.192371,0.8560
9,ICA_PROGRAM_18,-0.004868,-0.079060,0.071220,-0.096073,0.5420


In [14]:
# =============================================================================
# Define prespecified phenotype-sensitivity panels
# =============================================================================

NEAR_PRIMARY_PHENOTYPES = [
    "mean_ln_ic50",
    "mean_z_score",
    "median_z_score",
]

BROADER_PHENOTYPE_SENSITIVITIES = [
    "mean_auc",
    "median_auc",
]

PHENOTYPE_SENSITIVITY_PANEL = (
    NEAR_PRIMARY_PHENOTYPES
    + BROADER_PHENOTYPE_SENSITIVITIES
)

In [15]:
# =============================================================================
# Evaluate alternative-phenotype association sensitivity
# =============================================================================

phenotype_sensitivity_associations = []

for program_id in candidate_program_ids:
    program_residual = f"{program_id}_lineage_residual"

    for phenotype_name in PHENOTYPE_SENSITIVITY_PANEL:
        rho, _ = spearmanr(
            robustness_data[program_residual],
            robustness_data[
                f"{phenotype_name}_lineage_residual"
            ],
        )

        phenotype_sensitivity_associations.append(
            {
                "program_id": program_id,
                "phenotype_representation": phenotype_name,
                "spearman_rho": rho,
            }
        )

phenotype_sensitivity_associations = pd.DataFrame(
    phenotype_sensitivity_associations
)

phenotype_sensitivity_associations.pivot(
    index="program_id",
    columns="phenotype_representation",
    values="spearman_rho",
).loc[candidate_program_ids]

phenotype_representation,mean_auc,mean_ln_ic50,mean_z_score,median_auc,median_z_score
program_id,,,,,
ICA_PROGRAM_06,0.079305,0.154944,0.162070,0.031282,0.157481
ICA_PROGRAM_07,0.119135,0.112497,0.109030,0.087620,0.104814
ICA_PROGRAM_09,-0.045340,-0.052249,-0.045129,-0.019856,-0.048578
ICA_PROGRAM_13,-0.046976,-0.155432,-0.154779,0.035511,-0.166489
ICA_PROGRAM_18,0.038852,0.002801,0.004481,0.062662,0.008986
ICA_PROGRAM_20,0.063346,0.114992,0.115325,-0.012503,0.111346
ICA_PROGRAM_29,-0.046098,-0.082881,-0.077900,-0.036912,-0.080299
ICA_PROGRAM_33,-0.021432,-0.069065,-0.072022,-0.024316,-0.071747
ICA_PROGRAM_42,-0.007134,-0.055300,-0.061263,-0.013784,-0.062218


In [16]:
# =============================================================================
# Summarize alternative-phenotype association sensitivity
# =============================================================================

phenotype_sensitivity_summary = []

for program_id in candidate_program_ids:
    program_results = phenotype_sensitivity_associations.loc[
        phenotype_sensitivity_associations["program_id"].eq(program_id)
    ]

    discovery_sign = np.sign(
        candidate_programs.loc[
            candidate_programs["program_id"].eq(program_id),
            "spearman_rho",
        ].iloc[0]
    )

    near_primary = program_results.loc[
        program_results["phenotype_representation"].isin(
            NEAR_PRIMARY_PHENOTYPES
        )
    ]

    broader = program_results.loc[
        program_results["phenotype_representation"].isin(
            BROADER_PHENOTYPE_SENSITIVITIES
        )
    ]

    phenotype_sensitivity_summary.append(
        {
            "program_id": program_id,
            "near_primary_direction_preserved_fraction": (
                np.sign(near_primary["spearman_rho"])
                .eq(discovery_sign)
                .mean()
            ),
            "near_primary_min_abs_rho": (
                near_primary["spearman_rho"].abs().min()
            ),
            "broader_direction_preserved_fraction": (
                np.sign(broader["spearman_rho"])
                .eq(discovery_sign)
                .mean()
            ),
        }
    )

phenotype_sensitivity_summary = (
    pd.DataFrame(phenotype_sensitivity_summary)
    .sort_values(
        [
            "near_primary_direction_preserved_fraction",
            "near_primary_min_abs_rho",
        ],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

phenotype_sensitivity_summary

,program_id,near_primary_direction_preserved_fraction,near_primary_min_abs_rho,broader_direction_preserved_fraction
0,ICA_PROGRAM_18,0.0,0.002801,0.0
1,ICA_PROGRAM_09,1.0,0.045129,1.0
2,ICA_PROGRAM_42,1.0,0.055300,1.0
3,ICA_PROGRAM_33,1.0,0.069065,1.0
4,ICA_PROGRAM_29,1.0,0.077900,1.0
5,ICA_PROGRAM_07,1.0,0.104814,1.0
6,ICA_PROGRAM_20,1.0,0.111346,0.5
7,ICA_PROGRAM_13,1.0,0.154779,0.5
8,ICA_PROGRAM_06,1.0,0.154944,1.0
9,ICA_PROGRAM_46,1.0,0.254909,1.0


In [17]:
# =============================================================================
# Construct lineage-adjusted drug-response coverage
# =============================================================================

robustness_data[
    "drug_coverage_lineage_residual"
] = (
    robustness_data["n_unique_drug_ids"]
    - robustness_data
    .groupby("OncotreeLineage")["n_unique_drug_ids"]
    .transform("mean")
)

In [18]:
# =============================================================================
# Evaluate drug-response coverage sensitivity
# =============================================================================

coverage_adjusted_associations = []

coverage = robustness_data[
    "drug_coverage_lineage_residual"
].to_numpy()

design = np.column_stack(
    [
        np.ones(len(coverage)),
        coverage,
    ]
)

phenotype_residual = robustness_data[
    "selected_phenotype_lineage_residual"
].to_numpy()

phenotype_beta = np.linalg.lstsq(
    design,
    phenotype_residual,
    rcond=None,
)[0]

phenotype_coverage_residual = (
    phenotype_residual
    - design @ phenotype_beta
)

for program_id in candidate_program_ids:
    program_residual = robustness_data[
        f"{program_id}_lineage_residual"
    ].to_numpy()

    program_beta = np.linalg.lstsq(
        design,
        program_residual,
        rcond=None,
    )[0]

    program_coverage_residual = (
        program_residual
        - design @ program_beta
    )

    adjusted_rho, _ = spearmanr(
        program_coverage_residual,
        phenotype_coverage_residual,
    )

    lineage_only_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    coverage_adjusted_associations.append(
        {
            "program_id": program_id,
            "lineage_only_rho": lineage_only_rho,
            "lineage_coverage_adjusted_rho": adjusted_rho,
            "effect_retention": (
                abs(adjusted_rho) / abs(lineage_only_rho)
                if lineage_only_rho != 0
                else np.nan
            ),
        }
    )

coverage_adjusted_associations = (
    pd.DataFrame(coverage_adjusted_associations)
    .sort_values("effect_retention")
    .reset_index(drop=True)
)

coverage_adjusted_associations

,program_id,lineage_only_rho,lineage_coverage_adjusted_rho,effect_retention
0,ICA_PROGRAM_09,-0.038606,-0.033199,0.859942
1,ICA_PROGRAM_29,-0.097379,-0.086085,0.884021
2,ICA_PROGRAM_06,0.149012,0.139037,0.933057
3,ICA_PROGRAM_42,-0.051702,-0.049166,0.950943
4,ICA_PROGRAM_07,0.109690,0.107958,0.984203
5,ICA_PROGRAM_46,0.245619,0.248728,1.012658
6,ICA_PROGRAM_33,-0.075251,-0.077757,1.033291
7,ICA_PROGRAM_13,-0.136528,-0.141496,1.036389
8,ICA_PROGRAM_20,0.099840,0.104028,1.041950
9,ICA_PROGRAM_18,-0.000377,0.003593,9.529037


In [19]:
# =============================================================================
# Cell-line metadata input paths
# =============================================================================

MODEL_METADATA_PATH = (
    Paths.depmap
    / "Model.csv"
)

DEFAULT_PROFILE_PATH = (
    Paths.depmap
    / "OmicsDefaultModelProfiles.csv"
)

OMICS_PROFILES_PATH = (
    Paths.depmap
    / "OmicsProfiles.csv"
)

In [20]:
# =============================================================================
# Load cell-line metadata
# =============================================================================

model_metadata = pd.read_csv(MODEL_METADATA_PATH)
default_profile_metadata = pd.read_csv(DEFAULT_PROFILE_PATH)
omics_profile_metadata = pd.read_csv(OMICS_PROFILES_PATH)

In [21]:
# =============================================================================
# Identify prioritized cell-line covariate columns
# =============================================================================

covariate_terms = [
    "sex",
    "primary",
    "metast",
    "age",
    "growth",
    "collection",
    "source",
    "medium",
    "formulation",
    "stranded",
    "modelid",
    "profile",
]

for name, table in {
    "Model.csv": model_metadata,
    "OmicsDefaultModelProfiles.csv": default_profile_metadata,
    "OmicsProfiles.csv": omics_profile_metadata,
}.items():
    matched_columns = [
        column
        for column in table.columns
        if any(
            term in column.lower()
            for term in covariate_terms
        )
    ]

    print(f"\n{name}")
    print(matched_columns)


Model.csv
['ModelID', 'OncotreeLineage', 'OncotreePrimaryDisease', 'Age', 'AgeCategory', 'Sex', 'PrimaryOrMetastasis', 'SampleCollectionSite', 'SourceType', 'SourceDetail', 'Stage', 'GrowthPattern', 'FormulationID', 'SangerModelID']

OmicsDefaultModelProfiles.csv
['ModelID', 'ProfileID', 'ProfileType']

OmicsProfiles.csv
['ProfileID', 'ModelID', 'Stranded', 'Source']


In [22]:
# =============================================================================
# Inspect RNA profile metadata categories
# =============================================================================

print("Default profile types:")
print(
    default_profile_metadata["ProfileType"]
    .value_counts(dropna=False)
)

print("\nRNA strandedness:")
print(
    omics_profile_metadata["Stranded"]
    .value_counts(dropna=False)
)

print("\nProfile sources:")
print(
    omics_profile_metadata["Source"]
    .value_counts(dropna=False)
)

Default profile types:
ProfileType
dna    1929
rna    1673
Name: count, dtype: int64

RNA strandedness:
Stranded
NaN      2828
False    1005
True      685
Name: count, dtype: int64

Profile sources:
Source
BROAD     3492
SANGER    1026
Name: count, dtype: int64


In [23]:
# =============================================================================
# Construct default RNA-profile metadata
# =============================================================================

rna_profile_metadata = (
    default_profile_metadata
    .loc[
        default_profile_metadata["ProfileType"].eq("rna"),
        ["ModelID", "ProfileID"],
    ]
    .merge(
        omics_profile_metadata[
            ["ProfileID", "ModelID", "Stranded", "Source"]
        ],
        on=["ProfileID", "ModelID"],
        how="left",
    )
    .rename(
        columns={
            "Stranded": "RNA_Stranded",
            "Source": "RNA_Source",
        }
    )
)

rna_profile_metadata.head()

,ModelID,ProfileID,RNA_Stranded,RNA_Source
0,ACH-000873,PR-kl468e,False,BROAD
1,ACH-000860,PR-yL4kGI,False,BROAD
2,ACH-001715,PR-pUWMlI,True,BROAD
3,ACH-001627,PR-2WACfU,True,BROAD
4,ACH-000439,PR-tAgQgg,False,BROAD


In [24]:
# =============================================================================
# Construct candidate covariate table
# =============================================================================

candidate_covariates = [
    "Age",
    "Sex",
    "PrimaryOrMetastasis",
    "SampleCollectionSite",
    "SourceType",
    "GrowthPattern",
    "FormulationID",
]

cellline_covariates = (
    robustness_data[["ModelID", "OncotreeLineage"]]
    .merge(
        model_metadata[
            ["ModelID", *candidate_covariates]
        ],
        on="ModelID",
        how="left",
    )
    .merge(
        rna_profile_metadata[
            ["ModelID", "RNA_Stranded", "RNA_Source"]
        ],
        on="ModelID",
        how="left",
    )
)

covariate_inventory = pd.DataFrame(
    {
        "non_missing": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).notna().sum(),
        "coverage_fraction": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).notna().mean(),
        "n_unique": cellline_covariates.drop(
            columns=["ModelID", "OncotreeLineage"]
        ).nunique(dropna=True),
    }
).sort_values(
    ["coverage_fraction", "n_unique"],
    ascending=[False, True],
)

covariate_inventory

,non_missing,coverage_fraction,n_unique
RNA_Source,713,1.000000,1
RNA_Stranded,713,1.000000,2
Sex,713,1.000000,3
GrowthPattern,713,1.000000,3
SourceType,713,1.000000,11
SampleCollectionSite,713,1.000000,38
FormulationID,679,0.952314,84
PrimaryOrMetastasis,675,0.946704,2
Age,652,0.914446,88


In [25]:
# =============================================================================
# Assess pre-outcome covariate estimability
# =============================================================================

COVARIATE_TYPES = {
    "Sex": "categorical",
    "Age": "continuous",
    "PrimaryOrMetastasis": "categorical",
    "GrowthPattern": "categorical",
    "SampleCollectionSite": "categorical",
    "SourceType": "categorical",
    "FormulationID": "categorical",
    "RNA_Stranded": "categorical",
    "RNA_Source": "categorical",
}

covariate_estimability = []

for covariate, covariate_type in COVARIATE_TYPES.items():
    analysis_data = (
        cellline_covariates[
            ["OncotreeLineage", covariate]
        ]
        .dropna()
        .copy()
    )

    lineage_design = pd.get_dummies(
        analysis_data["OncotreeLineage"],
        drop_first=True,
        dtype=float,
    )

    baseline_design = np.column_stack(
        [
            np.ones(len(analysis_data)),
            lineage_design.to_numpy(),
        ]
    )

    if covariate_type == "continuous":
        covariate_design = (
            analysis_data[[covariate]]
            .astype(float)
            .to_numpy()
        )
    else:
        covariate_design = pd.get_dummies(
            analysis_data[covariate],
            drop_first=True,
            dtype=float,
        ).to_numpy()

    combined_design = np.column_stack(
        [
            baseline_design,
            covariate_design,
        ]
    )

    baseline_rank = np.linalg.matrix_rank(
        baseline_design
    )
    combined_rank = np.linalg.matrix_rank(
        combined_design
    )

    additional_rank = (
        combined_rank - baseline_rank
    )

    covariate_estimability.append(
        {
            "covariate": covariate,
            "coverage_fraction": (
                len(analysis_data)
                / len(cellline_covariates)
            ),
            "additional_rank": additional_rank,
            "design_full_rank": (
                combined_rank
                == combined_design.shape[1]
            ),
            "observations_per_added_parameter": (
                len(analysis_data) / additional_rank
                if additional_rank > 0
                else np.nan
            ),
            "residual_df": (
                len(analysis_data) - combined_rank
            ),
        }
    )

covariate_estimability = pd.DataFrame(
    covariate_estimability
)

covariate_estimability["joint_test_eligible"] = (
    covariate_estimability["coverage_fraction"].ge(0.80)
    & covariate_estimability["additional_rank"].gt(0)
    & covariate_estimability["design_full_rank"]
    & covariate_estimability[
        "observations_per_added_parameter"
    ].ge(10)
    & covariate_estimability["residual_df"].gt(0)
)

covariate_estimability

,covariate,coverage_fraction,additional_rank,design_full_rank,observations_per_added_parameter,residual_df,joint_test_eligible
0,Sex,1.000000,2,True,356.500000,684,True
1,Age,0.914446,1,True,652.000000,624,True
2,PrimaryOrMetastasis,0.946704,1,True,675.000000,647,True
3,GrowthPattern,1.000000,2,True,356.500000,684,True
4,SampleCollectionSite,1.000000,35,False,20.371429,651,False
5,SourceType,1.000000,10,True,71.300000,676,True
6,FormulationID,0.952314,83,True,8.180723,569,False
7,RNA_Stranded,1.000000,1,True,713.000000,685,True
8,RNA_Source,1.000000,0,True,NaN,686,False


In [26]:
# =============================================================================
# Freeze individual and joint covariate sensitivity sets
# =============================================================================

INDIVIDUAL_COVARIATES = [
    "Sex",
    "Age",
    "PrimaryOrMetastasis",
    "GrowthPattern",
    "SampleCollectionSite",
    "SourceType",
    "FormulationID",
    "RNA_Stranded",
]

JOINT_COVARIATES = (
    covariate_estimability
    .loc[
        covariate_estimability["joint_test_eligible"],
        "covariate",
    ]
    .tolist()
)

print("Individual sensitivity covariates:")
print(INDIVIDUAL_COVARIATES)

print("\nJoint-adjustment covariates:")
print(JOINT_COVARIATES)

Individual sensitivity covariates:
['Sex', 'Age', 'PrimaryOrMetastasis', 'GrowthPattern', 'SampleCollectionSite', 'SourceType', 'FormulationID', 'RNA_Stranded']

Joint-adjustment covariates:
['Sex', 'Age', 'PrimaryOrMetastasis', 'GrowthPattern', 'SourceType', 'RNA_Stranded']


In [27]:
# =============================================================================
# Residualization helper
# =============================================================================

def residualize(values, design_matrix):
    design = np.column_stack(
        [
            np.ones(len(design_matrix)),
            np.asarray(design_matrix, dtype=float),
        ]
    )

    coefficients = np.linalg.lstsq(
        design,
        np.asarray(values, dtype=float),
        rcond=None,
    )[0]

    return (
        np.asarray(values, dtype=float)
        - design @ coefficients
    )

In [28]:
# =============================================================================
# Evaluate individual covariate sensitivity
# =============================================================================

covariate_analysis_data = (
    robustness_data
    .merge(
        cellline_covariates.drop(
            columns="OncotreeLineage"
        ),
        on="ModelID",
        how="left",
    )
)

individual_covariate_results = []

for covariate in INDIVIDUAL_COVARIATES:
    covariate_type = COVARIATE_TYPES[covariate]

    for program_id in candidate_program_ids:
        analysis_data = (
            covariate_analysis_data[
                [
                    "OncotreeLineage",
                    "selected_phenotype",
                    program_id,
                    covariate,
                ]
            ]
            .dropna()
            .copy()
        )

        lineage_design = pd.get_dummies(
            analysis_data["OncotreeLineage"],
            drop_first=True,
            dtype=float,
        )

        if covariate_type == "continuous":
            covariate_design = (
                analysis_data[[covariate]]
                .astype(float)
            )
        else:
            covariate_design = pd.get_dummies(
                analysis_data[covariate],
                drop_first=True,
                dtype=float,
            )

        combined_design = pd.concat(
            [
                lineage_design.reset_index(drop=True),
                covariate_design.reset_index(drop=True),
            ],
            axis=1,
        )

        program_lineage_residual = residualize(
            analysis_data[program_id],
            lineage_design,
        )

        phenotype_lineage_residual = residualize(
            analysis_data["selected_phenotype"],
            lineage_design,
        )

        program_adjusted_residual = residualize(
            analysis_data[program_id],
            combined_design,
        )

        phenotype_adjusted_residual = residualize(
            analysis_data["selected_phenotype"],
            combined_design,
        )

        baseline_rho, _ = spearmanr(
            program_lineage_residual,
            phenotype_lineage_residual,
        )

        adjusted_rho, _ = spearmanr(
            program_adjusted_residual,
            phenotype_adjusted_residual,
        )

        program_partial_r2 = (
            1
            - np.sum(program_adjusted_residual ** 2)
            / np.sum(program_lineage_residual ** 2)
        )

        individual_covariate_results.append(
            {
                "program_id": program_id,
                "covariate": covariate,
                "n_models": len(analysis_data),
                "lineage_only_rho": baseline_rho,
                "adjusted_rho": adjusted_rho,
                "effect_retention": (
                    abs(adjusted_rho) / abs(baseline_rho)
                    if baseline_rho != 0
                    else np.nan
                ),
                "program_partial_r2": program_partial_r2,
            }
        )

individual_covariate_results = pd.DataFrame(
    individual_covariate_results
)

print("Individual covariate comparisons:", individual_covariate_results.shape)

Individual covariate comparisons: (80, 7)


In [29]:
# =============================================================================
# Summarize individual covariate sensitivity
# =============================================================================

individual_covariate_results[
    "direction_preserved"
] = (
    np.sign(individual_covariate_results["adjusted_rho"])
    == np.sign(individual_covariate_results["lineage_only_rho"])
)

individual_covariate_summary = []

for program_id in candidate_program_ids:
    program_results = individual_covariate_results.loc[
        individual_covariate_results["program_id"].eq(program_id)
    ].copy()

    most_attenuating = program_results.loc[
        program_results["effect_retention"].idxmin()
    ]

    strongest_covariate = program_results.loc[
        program_results["program_partial_r2"].idxmax()
    ]

    individual_covariate_summary.append(
        {
            "program_id": program_id,
            "min_effect_retention": (
                most_attenuating["effect_retention"]
            ),
            "most_attenuating_covariate": (
                most_attenuating["covariate"]
            ),
            "direction_preserved_fraction": (
                program_results["direction_preserved"].mean()
            ),
            "max_program_partial_r2": (
                strongest_covariate["program_partial_r2"]
            ),
            "strongest_program_covariate": (
                strongest_covariate["covariate"]
            ),
        }
    )

individual_covariate_summary = (
    pd.DataFrame(individual_covariate_summary)
    .sort_values(
        [
            "direction_preserved_fraction",
            "min_effect_retention",
        ]
    )
    .reset_index(drop=True)
)

individual_covariate_summary

,program_id,min_effect_retention,most_attenuating_covariate,direction_preserved_fraction,max_program_partial_r2,strongest_program_covariate
0,ICA_PROGRAM_18,0.745960,PrimaryOrMetastasis,0.375,0.068616,FormulationID
1,ICA_PROGRAM_09,0.584121,PrimaryOrMetastasis,1.000,0.129168,SampleCollectionSite
2,ICA_PROGRAM_33,0.599946,FormulationID,1.000,0.041669,FormulationID
3,ICA_PROGRAM_29,0.673341,FormulationID,1.000,0.135651,FormulationID
4,ICA_PROGRAM_07,0.842284,FormulationID,1.000,0.102447,FormulationID
5,ICA_PROGRAM_06,0.868532,SourceType,1.000,0.119708,FormulationID
6,ICA_PROGRAM_20,0.876062,GrowthPattern,1.000,0.088101,FormulationID
7,ICA_PROGRAM_42,0.884631,FormulationID,1.000,0.132806,FormulationID
8,ICA_PROGRAM_46,0.979653,SourceType,1.000,0.096976,FormulationID
9,ICA_PROGRAM_13,0.984790,Sex,1.000,0.135743,FormulationID


In [30]:
# =============================================================================
# Evaluate joint covariate sensitivity
# =============================================================================

joint_analysis_data = (
    covariate_analysis_data[
        [
            "OncotreeLineage",
            "selected_phenotype",
            *candidate_program_ids,
            *JOINT_COVARIATES,
        ]
    ]
    .dropna()
    .copy()
)

lineage_design = pd.get_dummies(
    joint_analysis_data["OncotreeLineage"],
    drop_first=True,
    dtype=float,
)

joint_covariate_blocks = []

for covariate in JOINT_COVARIATES:
    if COVARIATE_TYPES[covariate] == "continuous":
        block = joint_analysis_data[
            [covariate]
        ].astype(float)
    else:
        block = pd.get_dummies(
            joint_analysis_data[covariate],
            prefix=covariate,
            drop_first=True,
            dtype=float,
        )

    joint_covariate_blocks.append(
        block.reset_index(drop=True)
    )

joint_covariate_design = pd.concat(
    joint_covariate_blocks,
    axis=1,
)

combined_design = pd.concat(
    [
        lineage_design.reset_index(drop=True),
        joint_covariate_design,
    ],
    axis=1,
)

phenotype_lineage_residual = residualize(
    joint_analysis_data["selected_phenotype"],
    lineage_design,
)

phenotype_joint_residual = residualize(
    joint_analysis_data["selected_phenotype"],
    combined_design,
)

joint_covariate_results = []

for program_id in candidate_program_ids:
    program_lineage_residual = residualize(
        joint_analysis_data[program_id],
        lineage_design,
    )

    program_joint_residual = residualize(
        joint_analysis_data[program_id],
        combined_design,
    )

    baseline_rho, _ = spearmanr(
        program_lineage_residual,
        phenotype_lineage_residual,
    )

    adjusted_rho, _ = spearmanr(
        program_joint_residual,
        phenotype_joint_residual,
    )

    joint_covariate_results.append(
        {
            "program_id": program_id,
            "n_models": len(joint_analysis_data),
            "lineage_only_rho": baseline_rho,
            "joint_adjusted_rho": adjusted_rho,
            "effect_retention": (
                abs(adjusted_rho) / abs(baseline_rho)
                if baseline_rho != 0
                else np.nan
            ),
            "program_partial_r2": (
                1
                - np.sum(program_joint_residual ** 2)
                / np.sum(program_lineage_residual ** 2)
            ),
        }
    )

joint_covariate_results = (
    pd.DataFrame(joint_covariate_results)
    .sort_values("effect_retention")
    .reset_index(drop=True)
)

joint_covariate_results

,program_id,n_models,lineage_only_rho,joint_adjusted_rho,effect_retention,program_partial_r2
0,ICA_PROGRAM_09,615,-0.039850,-0.028963,0.726803,0.082075
1,ICA_PROGRAM_42,615,-0.052423,-0.039882,0.760773,0.052592
2,ICA_PROGRAM_07,615,0.101154,0.082203,0.812652,0.023924
3,ICA_PROGRAM_33,615,-0.061571,-0.055955,0.908781,0.041833
4,ICA_PROGRAM_46,615,0.234057,0.219464,0.937650,0.019650
5,ICA_PROGRAM_06,615,0.176528,0.167309,0.947772,0.050258
6,ICA_PROGRAM_29,615,-0.103557,-0.101539,0.980510,0.031130
7,ICA_PROGRAM_20,615,0.125864,0.126786,1.007328,0.072267
8,ICA_PROGRAM_13,615,-0.134432,-0.156785,1.166277,0.026549
9,ICA_PROGRAM_18,615,0.028626,0.050840,1.776009,0.038372


In [31]:
# =============================================================================
# Compute lineage-preserving phenotype permutation nulls
# =============================================================================

N_PERMUTATIONS = 2000
PERMUTATION_SEED = RANDOM_SEED + 1

permutation_rng = np.random.default_rng(
    PERMUTATION_SEED
)

lineage_positions = list(
    robustness_data
    .groupby("OncotreeLineage", sort=False)
    .indices
    .values()
)

phenotype_residual = robustness_data[
    "selected_phenotype_lineage_residual"
].to_numpy()

program_residuals = {
    program_id: robustness_data[
        f"{program_id}_lineage_residual"
    ].to_numpy()
    for program_id in candidate_program_ids
}

permutation_associations = []

for permutation_id in range(N_PERMUTATIONS):
    permuted_phenotype = phenotype_residual.copy()

    for positions in lineage_positions:
        permuted_phenotype[positions] = (
            permutation_rng.permutation(
                permuted_phenotype[positions]
            )
        )

    for program_id in candidate_program_ids:
        rho, _ = spearmanr(
            program_residuals[program_id],
            permuted_phenotype,
        )

        permutation_associations.append(
            {
                "permutation_id": permutation_id,
                "program_id": program_id,
                "spearman_rho": rho,
            }
        )

permutation_associations = pd.DataFrame(
    permutation_associations
)

print(
    "Lineage-preserving permutations completed:",
    permutation_associations["permutation_id"].nunique(),
)

Lineage-preserving permutations completed: 2000


In [32]:
# =============================================================================
# Summarize lineage-preserving permutation controls
# =============================================================================

permutation_summary = []

for program_id in candidate_program_ids:
    null_rhos = permutation_associations.loc[
        permutation_associations["program_id"].eq(program_id),
        "spearman_rho",
    ]

    observed_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    exceedance_count = (
        null_rhos.abs() >= abs(observed_rho)
    ).sum()

    permutation_summary.append(
        {
            "program_id": program_id,
            "observed_rho": observed_rho,
            "null_median": null_rhos.median(),
            "null_ci_lower": null_rhos.quantile(0.025),
            "null_ci_upper": null_rhos.quantile(0.975),
            "null_abs_95th_percentile": (
                null_rhos.abs().quantile(0.95)
            ),
            "null_exceedance_fraction": (
                (exceedance_count + 1)
                / (N_PERMUTATIONS + 1)
            ),
        }
    )

permutation_summary = (
    pd.DataFrame(permutation_summary)
    .sort_values(
        "observed_rho",
        key=lambda x: x.abs(),
        ascending=False,
    )
    .reset_index(drop=True)
)

permutation_summary

,program_id,observed_rho,null_median,null_ci_lower,null_ci_upper,null_abs_95th_percentile,null_exceedance_fraction
0,ICA_PROGRAM_46,0.245619,-0.000156,-0.073262,0.070386,0.071814,0.000500
1,ICA_PROGRAM_06,0.149012,-0.002600,-0.076221,0.071982,0.074518,0.000500
2,ICA_PROGRAM_13,-0.136528,0.000888,-0.073743,0.072811,0.073276,0.000500
3,ICA_PROGRAM_07,0.109690,0.002390,-0.073107,0.081108,0.076669,0.005497
4,ICA_PROGRAM_20,0.099840,-0.000919,-0.076449,0.075387,0.076395,0.010495
5,ICA_PROGRAM_29,-0.097379,-0.001476,-0.077566,0.072242,0.074431,0.008496
6,ICA_PROGRAM_33,-0.075251,-0.000079,-0.070261,0.066454,0.068435,0.032984
7,ICA_PROGRAM_42,-0.051702,-0.002876,-0.072957,0.065945,0.069487,0.151424
8,ICA_PROGRAM_09,-0.038606,-0.000097,-0.070130,0.074600,0.072430,0.297851
9,ICA_PROGRAM_18,-0.000377,-0.000381,-0.077438,0.074546,0.075353,0.992004


In [33]:
# =============================================================================
# Derive leave-one-lineage-out influence flags
# =============================================================================

lolo_influence_summary = []

for program_id in candidate_program_ids:
    program_lolo = lolo_associations.loc[
        lolo_associations["program_id"].eq(program_id)
    ].copy()

    full_rho = lineage_adjusted_associations.loc[
        lineage_adjusted_associations["program_id"].eq(program_id),
        "lineage_adjusted_rho",
    ].iloc[0]

    sign_reversal = (
        np.sign(program_lolo["spearman_rho"])
        != np.sign(full_rho)
    ).any()

    minimum_effect_retention = (
        program_lolo["spearman_rho"].abs().min()
        / abs(full_rho)
        if full_rho != 0
        else np.nan
    )

    lolo_influence_summary.append(
        {
            "program_id": program_id,
            "minimum_lolo_effect_retention": (
                minimum_effect_retention
            ),
            "lolo_sign_reversal": sign_reversal,
            "single_lineage_influenced": (
                sign_reversal
                or minimum_effect_retention < 0.50
            ),
        }
    )

lolo_influence_summary = (
    pd.DataFrame(lolo_influence_summary)
    .sort_values("minimum_lolo_effect_retention")
    .reset_index(drop=True)
)

lolo_influence_summary

,program_id,minimum_lolo_effect_retention,lolo_sign_reversal,single_lineage_influenced
0,ICA_PROGRAM_18,0.177472,True,True
1,ICA_PROGRAM_09,0.187390,False,True
2,ICA_PROGRAM_42,0.522905,False,False
3,ICA_PROGRAM_33,0.663793,False,False
4,ICA_PROGRAM_20,0.735075,False,False
5,ICA_PROGRAM_29,0.859904,False,False
6,ICA_PROGRAM_13,0.867497,False,False
7,ICA_PROGRAM_46,0.877381,False,False
8,ICA_PROGRAM_07,0.890732,False,False
9,ICA_PROGRAM_06,0.918914,False,False


In [34]:
# =============================================================================
# Integrate prespecified robustness flags
# =============================================================================

robustness_flags = (
    candidate_programs[
        ["program_id", "spearman_rho"]
    ]
    .rename(
        columns={"spearman_rho": "discovery_rho"}
    )
    .merge(
        lineage_adjusted_associations[
            [
                "program_id",
                "lineage_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention": "lineage_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        bootstrap_summary[
            [
                "program_id",
                "direction_preserved_fraction",
            ]
        ].rename(
            columns={
                "direction_preserved_fraction":
                    "bootstrap_direction_preserved_fraction"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        phenotype_sensitivity_summary[
            [
                "program_id",
                "near_primary_direction_preserved_fraction",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        lolo_influence_summary[
            [
                "program_id",
                "single_lineage_influenced",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        coverage_adjusted_associations[
            [
                "program_id",
                "lineage_coverage_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention":
                    "coverage_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        individual_covariate_summary[
            [
                "program_id",
                "min_effect_retention",
                "direction_preserved_fraction",
            ]
        ].rename(
            columns={
                "min_effect_retention":
                    "individual_covariate_min_retention",
                "direction_preserved_fraction":
                    "individual_covariate_direction_fraction",
            }
        ),
        on="program_id",
        how="left",
    )
    .merge(
        joint_covariate_results[
            [
                "program_id",
                "joint_adjusted_rho",
                "effect_retention",
            ]
        ].rename(
            columns={
                "effect_retention":
                    "joint_covariate_effect_retention"
            }
        ),
        on="program_id",
        how="left",
    )
)

robustness_flags["lineage_sensitive"] = (
    (
        np.sign(robustness_flags["lineage_adjusted_rho"])
        != np.sign(robustness_flags["discovery_rho"])
    )
    | robustness_flags["lineage_effect_retention"].lt(0.50)
)

robustness_flags["bootstrap_direction_unstable"] = (
    robustness_flags[
        "bootstrap_direction_preserved_fraction"
    ].lt(0.95)
)

robustness_flags["near_primary_phenotype_sensitive"] = (
    robustness_flags[
        "near_primary_direction_preserved_fraction"
    ].lt(1.0)
)

robustness_flags["coverage_sensitive"] = (
    (
        np.sign(
            robustness_flags["lineage_coverage_adjusted_rho"]
        )
        != np.sign(
            robustness_flags["lineage_adjusted_rho"]
        )
    )
    | robustness_flags[
        "coverage_effect_retention"
    ].lt(0.50)
)

robustness_flags["individual_covariate_sensitive"] = (
    robustness_flags[
        "individual_covariate_direction_fraction"
    ].lt(1.0)
    | robustness_flags[
        "individual_covariate_min_retention"
    ].lt(0.50)
)

robustness_flags["joint_covariate_sensitive"] = (
    (
        np.sign(robustness_flags["joint_adjusted_rho"])
        != np.sign(
            joint_covariate_results
            .set_index("program_id")
            .loc[
                robustness_flags["program_id"],
                "lineage_only_rho",
            ]
            .to_numpy()
        )
    )
    | robustness_flags[
        "joint_covariate_effect_retention"
    ].lt(0.50)
)

robustness_flags[
    [
        "program_id",
        "lineage_sensitive",
        "bootstrap_direction_unstable",
        "near_primary_phenotype_sensitive",
        "single_lineage_influenced",
        "coverage_sensitive",
        "individual_covariate_sensitive",
        "joint_covariate_sensitive",
    ]
]

,program_id,lineage_sensitive,bootstrap_direction_unstable,near_primary_phenotype_sensitive,single_lineage_influenced,coverage_sensitive,individual_covariate_sensitive,joint_covariate_sensitive
0,ICA_PROGRAM_06,False,False,False,False,False,False,False
1,ICA_PROGRAM_07,False,False,False,False,False,False,False
2,ICA_PROGRAM_09,True,True,False,True,False,False,False
3,ICA_PROGRAM_13,False,False,False,False,False,False,False
4,ICA_PROGRAM_18,True,True,True,True,True,True,False
5,ICA_PROGRAM_20,False,False,False,False,False,False,False
6,ICA_PROGRAM_29,False,False,False,False,False,False,False
7,ICA_PROGRAM_33,False,False,False,False,False,False,False
8,ICA_PROGRAM_42,True,True,False,False,False,False,False
9,ICA_PROGRAM_46,False,False,False,False,False,False,False


In [35]:
# =============================================================================
# Assign hierarchical robustness categories
# =============================================================================

robustness_flags["association_unstable"] = (
    robustness_flags[
        "near_primary_phenotype_sensitive"
    ]
)

robustness_flags["unresolved_confounding"] = (
    ~robustness_flags["association_unstable"]
    & (
        robustness_flags["coverage_sensitive"]
        | robustness_flags[
            "individual_covariate_sensitive"
        ]
        | robustness_flags[
            "joint_covariate_sensitive"
        ]
    )
)

robustness_flags["context_sensitive"] = (
    ~robustness_flags["association_unstable"]
    & ~robustness_flags["unresolved_confounding"]
    & (
        robustness_flags["lineage_sensitive"]
        | robustness_flags[
            "bootstrap_direction_unstable"
        ]
        | robustness_flags[
            "single_lineage_influenced"
        ]
    )
)

robustness_flags["robustness_category"] = np.select(
    [
        robustness_flags["association_unstable"],
        robustness_flags["unresolved_confounding"],
        robustness_flags["context_sensitive"],
    ],
    [
        "ASSOCIATION_UNSTABLE_CANDIDATE",
        "UNRESOLVED_CONFOUNDED_CANDIDATE",
        "CONTEXT_SENSITIVE_CANDIDATE",
    ],
    default="ROBUSTNESS_SUPPORTED_CANDIDATE",
)

robustness_flags[
    [
        "program_id",
        "robustness_category",
    ]
].sort_values(
    ["robustness_category", "program_id"]
)

,program_id,robustness_category
4,ICA_PROGRAM_18,ASSOCIATION_UNSTABLE_CANDIDATE
2,ICA_PROGRAM_09,CONTEXT_SENSITIVE_CANDIDATE
8,ICA_PROGRAM_42,CONTEXT_SENSITIVE_CANDIDATE
0,ICA_PROGRAM_06,ROBUSTNESS_SUPPORTED_CANDIDATE
1,ICA_PROGRAM_07,ROBUSTNESS_SUPPORTED_CANDIDATE
3,ICA_PROGRAM_13,ROBUSTNESS_SUPPORTED_CANDIDATE
5,ICA_PROGRAM_20,ROBUSTNESS_SUPPORTED_CANDIDATE
6,ICA_PROGRAM_29,ROBUSTNESS_SUPPORTED_CANDIDATE
7,ICA_PROGRAM_33,ROBUSTNESS_SUPPORTED_CANDIDATE
9,ICA_PROGRAM_46,ROBUSTNESS_SUPPORTED_CANDIDATE


## Integrated robustness interpretation

### Classification framework

Robustness categories are assigned hierarchically to the 10 discovery-level candidate programs frozen in notebook 310.

The classification is based on prespecified stress tests and does not redefine the candidate universe.

The hierarchy is:

1. `ASSOCIATION_UNSTABLE_CANDIDATE`
   - assigned when the association does not preserve its discovery direction across the frozen near-primary pharmacological representations;
   - this category takes precedence over apparent covariate or coverage sensitivity because effect-retention ratios and sign changes become unstable when the lineage-adjusted association is already approximately null.

2. `UNRESOLVED_CONFOUNDED_CANDIDATE`
   - assigned to an otherwise interpretable association showing a sign reversal or <50% effect retention after adjustment for drug-response coverage or prespecified cell-line covariates.

3. `CONTEXT_SENSITIVE_CANDIDATE`
   - assigned when the association is not classified as unstable or confounded but shows substantial lineage dependence, bootstrap directional instability, or material leave-one-lineage-out influence.

4. `ROBUSTNESS_SUPPORTED_CANDIDATE`
   - assigned when none of the preceding conditions is met.

These categories describe internal association robustness and must not be interpreted as independent validation or biological causality.

### Current classification

Seven programs are classified as `ROBUSTNESS_SUPPORTED_CANDIDATE`:

- `ICA_PROGRAM_06`
- `ICA_PROGRAM_07`
- `ICA_PROGRAM_13`
- `ICA_PROGRAM_20`
- `ICA_PROGRAM_29`
- `ICA_PROGRAM_33`
- `ICA_PROGRAM_46`

Two programs are classified as `CONTEXT_SENSITIVE_CANDIDATE`:

- `ICA_PROGRAM_09`
- `ICA_PROGRAM_42`

`ICA_PROGRAM_09` shows strong attenuation after lineage adjustment, reduced bootstrap directional stability, and material sensitivity to leaving out a single lineage. Its direction remains stable across near-primary phenotype representations and the association is not materially explained by the evaluated coverage or metadata covariates.

`ICA_PROGRAM_42` also shows substantial attenuation after lineage adjustment and falls slightly below the prespecified 95% bootstrap direction-preservation criterion. Its leave-one-lineage-out behavior does not cross the prespecified influence threshold, and no evaluated covariate materially explains the remaining association.

One program is classified as `ASSOCIATION_UNSTABLE_CANDIDATE`:

- `ICA_PROGRAM_18`

For `ICA_PROGRAM_18`, the discovery association is almost completely removed by lineage adjustment, bootstrap direction preservation is close to chance, leave-one-lineage-out analyses include sign reversals, and none of the frozen near-primary phenotype representations preserves the discovery direction. Apparent coverage or individual-covariate sensitivity for this program is not interpreted as evidence of specific confounding because its lineage-adjusted association is already approximately zero.

No program is currently classified as `UNRESOLVED_CONFOUNDED_CANDIDATE`.

### Covariate interpretation

Individual sensitivity analyses considered:

- sex;
- age;
- primary versus metastatic origin;
- growth pattern;
- sample collection site;
- source type;
- formulation;
- RNA strandedness.

The joint adjustment was restricted, before examining program-level effects, to covariates satisfying the prespecified estimability criteria. The eligible joint set was:

- sex;
- age;
- primary versus metastatic origin;
- growth pattern;
- source type;
- RNA strandedness.

`SampleCollectionSite` was retained for individual sensitivity analysis only because its combined design was not full rank after lineage adjustment.

`FormulationID` was retained for individual sensitivity analysis only because its high dimensionality did not satisfy the prespecified observations-per-added-parameter criterion.

RNA profile source was not evaluated as a robustness covariate because all 713 frozen models use BROAD RNA profiles and therefore provide no source variation.

No interpretable candidate association showed a sign reversal or <50% effect retention under the joint covariate stress test.

### Pharmacological ascertainment

Drug-response coverage was evaluated separately from biological and technical metadata.

No candidate with a non-negligible lineage-adjusted association showed material attenuation under additional coverage adjustment.

This analysis addresses response-coverage ascertainment only. It does not establish robustness to drug-family composition or screen-specific pharmacological structure.

### Negative-control interpretation

Lineage-preserving phenotype permutations were used as a negative control while retaining the observed lineage structure.

The strongest lineage-adjusted associations were clearly displaced from their corresponding permutation nulls, whereas `ICA_PROGRAM_42`, `ICA_PROGRAM_09`, and especially `ICA_PROGRAM_18` showed progressively weaker separation from the null distribution.

Permutation exceedance fractions are treated as diagnostic quantities rather than as a new candidate-selection or multiple-testing procedure. Candidate status remains frozen from notebook 310.

### Remaining limitations

These analyses are internal post-selection stress tests performed in the same 713-model cohort used for candidate discovery.

They do not constitute cross-dataset validation.

A frozen cell-line proliferation covariate was not identified upstream. No expression-derived proliferation score is constructed post hoc in this notebook; residual proliferation confounding therefore remains unresolved.

RNA-source/platform robustness cannot be evaluated because the frozen cohort contains no RNA-source variation. RNA strandedness provides only a limited technical sensitivity analysis.

Cross-method ICA–NMF support remains a separate descriptive axis and is not used as a mandatory robustness criterion.

Association robustness does not imply biological causality, clinical resistance prediction, or therapeutic relevance.

In [36]:
# =============================================================================
# Construct integrated candidate robustness summary
# =============================================================================

program_robustness_summary = (
    robustness_flags
    .merge(
        lineage_eta_squared,
        on="program_id",
        how="left",
    )
    .merge(
        lolo_summary[
            [
                "program_id",
                "most_influential_lineage",
                "max_absolute_deviation",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        bootstrap_summary[
            [
                "program_id",
                "bootstrap_median",
                "ci_lower",
                "ci_upper",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        permutation_summary[
            [
                "program_id",
                "null_abs_95th_percentile",
                "null_exceedance_fraction",
            ]
        ],
        on="program_id",
        how="left",
    )
    .merge(
        candidate_programs[
            [
                "program_id",
                "cross_method_convergent",
                "program_status",
            ]
        ],
        on="program_id",
        how="left",
    )
)

program_robustness_summary[
    [
        "program_id",
        "robustness_category",
        "discovery_rho",
        "lineage_adjusted_rho",
        "lineage_effect_retention",
        "lineage_eta_squared",
        "bootstrap_direction_preserved_fraction",
        "most_influential_lineage",
        "null_exceedance_fraction",
        "cross_method_convergent",
        "program_status",
    ]
].sort_values(
    [
        "robustness_category",
        "program_id",
    ]
)

,program_id,robustness_category,discovery_rho,lineage_adjusted_rho,lineage_effect_retention,lineage_eta_squared,bootstrap_direction_preserved_fraction,most_influential_lineage,null_exceedance_fraction,cross_method_convergent,program_status
4,ICA_PROGRAM_18,ASSOCIATION_UNSTABLE_CANDIDATE,-0.096073,-0.000377,0.003925,0.555058,0.5420,Lung,0.992004,True,candidate_with_cross_method_support
2,ICA_PROGRAM_09,CONTEXT_SENSITIVE_CANDIDATE,-0.192371,-0.038606,0.200686,0.303030,0.8560,Breast,0.297851,True,candidate_with_cross_method_support
8,ICA_PROGRAM_42,CONTEXT_SENSITIVE_CANDIDATE,-0.132021,-0.051702,0.391622,0.137175,0.9475,Lung,0.151424,True,candidate_with_cross_method_support
0,ICA_PROGRAM_06,ROBUSTNESS_SUPPORTED_CANDIDATE,0.129352,0.149012,1.151987,0.242616,0.9995,Esophagus/Stomach,0.000500,True,candidate_with_cross_method_support
1,ICA_PROGRAM_07,ROBUSTNESS_SUPPORTED_CANDIDATE,0.100847,0.109690,1.087690,0.104019,0.9980,Breast,0.005497,False,candidate_ica_specific
3,ICA_PROGRAM_13,ROBUSTNESS_SUPPORTED_CANDIDATE,-0.112589,-0.136528,1.212625,0.075770,1.0000,Head and Neck,0.000500,False,candidate_ica_specific
5,ICA_PROGRAM_20,ROBUSTNESS_SUPPORTED_CANDIDATE,0.103747,0.099840,0.962338,0.118799,0.9960,Bowel,0.010495,True,candidate_with_cross_method_support
6,ICA_PROGRAM_29,ROBUSTNESS_SUPPORTED_CANDIDATE,-0.132617,-0.097379,0.734284,0.450314,0.9910,Lymphoid,0.008496,True,candidate_with_cross_method_support
7,ICA_PROGRAM_33,ROBUSTNESS_SUPPORTED_CANDIDATE,-0.145618,-0.075251,0.516772,0.107989,0.9710,Lymphoid,0.032984,True,candidate_with_cross_method_support
9,ICA_PROGRAM_46,ROBUSTNESS_SUPPORTED_CANDIDATE,0.211964,0.245619,1.158776,0.020955,1.0000,Lung,0.000500,False,candidate_ica_specific


In [37]:
# =============================================================================
# Robustness output paths
# =============================================================================

ROBUSTNESS_SUMMARY_PATH = (
    OUTPUT_DIR
    / "311_program_robustness_summary.csv"
)

LOLO_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_leave_one_lineage_out_associations.csv"
)

BOOTSTRAP_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_lineage_stratified_bootstrap_associations.parquet"
)

PHENOTYPE_SENSITIVITY_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_alternative_phenotype_associations.csv"
)

COVARIATE_SENSITIVITY_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_individual_covariate_sensitivity.csv"
)

JOINT_COVARIATE_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_joint_covariate_sensitivity.csv"
)

PERMUTATION_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_lineage_preserving_permutation_associations.parquet"
)

ROBUSTNESS_METADATA_PATH = (
    OUTPUT_DIR
    / "311_program_robustness_metadata.json"
)

In [38]:
# =============================================================================
# Write program-robustness analytical artifacts
# =============================================================================

COVARIATE_ESTIMABILITY_PATH = (
    OUTPUT_DIR
    / "311_covariate_estimability.csv"
)

COVERAGE_SENSITIVITY_RESULTS_PATH = (
    OUTPUT_DIR
    / "311_drug_coverage_sensitivity.csv"
)

program_robustness_summary.to_csv(
    ROBUSTNESS_SUMMARY_PATH,
    index=False,
)

lolo_associations.to_csv(
    LOLO_RESULTS_PATH,
    index=False,
)

bootstrap_associations.to_parquet(
    BOOTSTRAP_RESULTS_PATH,
    index=False,
)

phenotype_sensitivity_associations.to_csv(
    PHENOTYPE_SENSITIVITY_RESULTS_PATH,
    index=False,
)

covariate_estimability.to_csv(
    COVARIATE_ESTIMABILITY_PATH,
    index=False,
)

individual_covariate_results.to_csv(
    COVARIATE_SENSITIVITY_RESULTS_PATH,
    index=False,
)

joint_covariate_results.to_csv(
    JOINT_COVARIATE_RESULTS_PATH,
    index=False,
)

coverage_adjusted_associations.to_csv(
    COVERAGE_SENSITIVITY_RESULTS_PATH,
    index=False,
)

permutation_associations.to_parquet(
    PERMUTATION_RESULTS_PATH,
    index=False,
)

print("Program-robustness analytical artifacts written")
print(
    "Directory:",
    project_relative_path(OUTPUT_DIR),
)

Program-robustness analytical artifacts written
Directory: data/processed/cellline_programs


In [39]:
# =============================================================================
# Construct program-robustness metadata
# =============================================================================

robustness_metadata = {
    "notebook": "311_program_robustness",
    "analysis_scope": {
        "candidate_program_count": len(candidate_program_ids),
        "candidate_program_ids": candidate_program_ids,
        "primary_phenotype": "selected_phenotype",
        "cohort_model_count": len(robustness_data),
        "interpretation": (
            "Internal post-selection association-robustness analysis; "
            "not independent validation."
        ),
    },
    "resampling": {
        "bootstrap_iterations": N_BOOTSTRAP,
        "bootstrap_seed": RANDOM_SEED,
        "permutation_iterations": N_PERMUTATIONS,
        "permutation_seed": PERMUTATION_SEED,
        "strategy": "lineage-stratified",
    },
    "phenotype_sensitivity": {
        "near_primary_representations": NEAR_PRIMARY_PHENOTYPES,
        "broader_representations": BROADER_PHENOTYPE_SENSITIVITIES,
    },
    "covariates": {
        "individual": INDIVIDUAL_COVARIATES,
        "joint": JOINT_COVARIATES,
        "not_evaluable": {
            "RNA_Source": (
                "No variation among the frozen 713 models; "
                "all default RNA profiles originate from BROAD."
            ),
            "proliferation": (
                "No frozen upstream cell-line proliferation covariate "
                "was identified; no post hoc expression-derived score "
                "was constructed."
            ),
        },
    },
    "prespecified_thresholds": {
        "minimum_effect_retention": 0.50,
        "minimum_bootstrap_direction_preservation": 0.95,
        "minimum_covariate_coverage_for_joint_test": 0.80,
        "minimum_observations_per_added_parameter": 10,
    },
    "classification_hierarchy": [
        "ASSOCIATION_UNSTABLE_CANDIDATE",
        "UNRESOLVED_CONFOUNDED_CANDIDATE",
        "CONTEXT_SENSITIVE_CANDIDATE",
        "ROBUSTNESS_SUPPORTED_CANDIDATE",
    ],
    "classification_rules": {
        "ASSOCIATION_UNSTABLE_CANDIDATE": (
            "Near-primary phenotype direction is not fully preserved."
        ),
        "UNRESOLVED_CONFOUNDED_CANDIDATE": (
            "Otherwise interpretable association shows sign reversal "
            "or <50% effect retention under coverage or covariate adjustment."
        ),
        "CONTEXT_SENSITIVE_CANDIDATE": (
            "Association is not unstable or confounded but shows "
            "lineage sensitivity, bootstrap directional instability, "
            "or material leave-one-lineage-out influence."
        ),
        "ROBUSTNESS_SUPPORTED_CANDIDATE": (
            "None of the preceding robustness conditions is met."
        ),
    },
    "program_categories": (
        program_robustness_summary
        .set_index("program_id")["robustness_category"]
        .to_dict()
    ),
    "limitations": [
        (
            "Candidate programs were selected in the same cohort used "
            "for these robustness analyses."
        ),
        (
            "Permutation controls are diagnostic and are not used "
            "as a new candidate-selection procedure."
        ),
        (
            "Drug-response coverage adjustment does not address "
            "drug-family composition or cross-screen robustness."
        ),
        (
            "RNA platform/source robustness cannot be evaluated because "
            "all frozen models use BROAD default RNA profiles."
        ),
        (
            "RNA strandedness provides only a limited technical "
            "sensitivity analysis."
        ),
        (
            "Association robustness does not imply causality, clinical "
            "resistance prediction, or therapeutic relevance."
        ),
    ],
}

with ROBUSTNESS_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        robustness_metadata,
        handle,
        indent=2,
    )

print(
    "Metadata written:",
    project_relative_path(ROBUSTNESS_METADATA_PATH),
)

Metadata written: data/processed/cellline_programs/311_program_robustness_metadata.json


## Robustness summary and downstream interpretation

The 10 discovery-level candidate programs from notebook 310 were evaluated under lineage-aware and prespecified association-robustness stress tests.

### Final robustness categories

Seven candidates are classified as `ROBUSTNESS_SUPPORTED_CANDIDATE`:

- `ICA_PROGRAM_06`
- `ICA_PROGRAM_07`
- `ICA_PROGRAM_13`
- `ICA_PROGRAM_20`
- `ICA_PROGRAM_29`
- `ICA_PROGRAM_33`
- `ICA_PROGRAM_46`

Two candidates are classified as `CONTEXT_SENSITIVE_CANDIDATE`:

- `ICA_PROGRAM_09`
- `ICA_PROGRAM_42`

One candidate is classified as `ASSOCIATION_UNSTABLE_CANDIDATE`:

- `ICA_PROGRAM_18`

No candidate is classified as `UNRESOLVED_CONFOUNDED_CANDIDATE`.

### Main findings

Lineage adjustment substantially changed the interpretation of three discovery associations.

`ICA_PROGRAM_18` was almost completely attenuated after lineage adjustment and subsequently showed poor bootstrap directional stability, leave-one-lineage-out sign instability, failure across near-primary phenotype representations, and no separation from the lineage-preserving permutation null. Its discovery-level pharmacological association should therefore not be promoted downstream as a robust resistance-like association.

`ICA_PROGRAM_09` retained only a minority of its discovery effect after lineage adjustment and showed both bootstrap instability and material leave-one-lineage-out influence. Its remaining association is therefore interpreted as context-sensitive rather than broadly robust.

`ICA_PROGRAM_42` also showed substantial attenuation after lineage adjustment and marginally failed the prespecified bootstrap direction-preservation criterion, but it did not show material single-lineage influence or sensitivity to the evaluated covariates.

The remaining seven candidates preserved interpretable direction and magnitude across the primary lineage adjustment and the major prespecified sensitivity analyses.

### Covariate and technical sensitivity

No interpretable candidate association showed a sign reversal or loss of more than 50% of its lineage-adjusted effect under:

- drug-response coverage adjustment;
- any individual evaluable biological, provenance, culture, or technical covariate;
- the prespecified joint covariate adjustment.

Several program scores showed measurable association with formulation, collection site, source type, or other metadata. These associations describe variation in the transcriptomic representations but did not materially account for the corresponding resistance-like phenotype associations under the prespecified robustness criteria.

RNA-source robustness could not be evaluated because all default RNA profiles in the frozen cohort originated from BROAD. RNA strandedness therefore represents only a limited technical sensitivity analysis.

No frozen upstream proliferation covariate was available for the cell-line cohort. Residual proliferation confounding remains unresolved and no expression-derived proliferation score was constructed post hoc.

### Cross-method evidence

ICA–NMF convergence remains an independent descriptive property rather than a robustness gate.

In particular, cross-method support does not rescue an unstable pharmacological association, as illustrated by `ICA_PROGRAM_18`, and lack of cross-method convergence does not invalidate an otherwise robust ICA association, as illustrated by `ICA_PROGRAM_46`.

### Phase 4 handoff

All 10 frozen discovery-level candidates remain represented in the 311 outputs.

The robustness categories should be propagated to Phase 4 as metadata rather than used to retrospectively redefine the notebook 310 candidate universe.

Cross-system comparison must therefore distinguish:

- candidates with supported cell-line association robustness;
- candidates with context-sensitive association evidence;
- candidates whose discovery association became unstable under robustness testing.

Tumor–cell-line matching and consensus construction remain independent downstream analyses. No tumor-program identity or exploratory tumor ranking was used to define the cell-line robustness categories in this notebook.

These results represent internal post-selection computational associations. They do not establish independent validation, causal mechanisms, clinical resistance prediction, or therapeutic relevance.